In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer
import torch

model = AutoModelForTokenClassification.from_pretrained("Stoikopoulou/elcardiocc-mbert-ner-baseline")
tokenizer = AutoTokenizer.from_pretrained("Stoikopoulou/elcardiocc-mbert-ner-baseline")

In [ ]:
import warnings
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
from collections import Counter
import re

def get_ner_predictions(model, tokenizer, document, label_map, current_text):

    """
    NER Inference Pipeline: Subword-to-Entity Reconstruction

    This script processes raw model outputs into structured entities. 
    It handles the transition from BERT subwords to whole words, and 
    from words to merged phrases.

    ALGORITHM LOGIC:
    1. Subword Aggregation: Rebuilds words by merging Lexical Heads with 
    sub-units (prefixed with '##').
    2. Label Resolution: Determines the final label via Majority Voting 
    across word fragments.
    3. Coordinate Alignment: Maps words to character offsets using a 
    Stateful Pointer to ensure unique indexing of duplicate terms.
    4. Phrase Grouping: Merges adjacent words if they share a label and 
    maintain a 'Strict +1 Gap'.

    KNOWN DANGER ZONES:
    - Strict +1 Gap: Logic requires an exact 1-character distance. This 
    causes phrases to split if the text contains double spaces or newlines.
    - Character Mismatch: The tokenizer does modify symbols (/, (, +) and 
    Greek accents. These modifications cause Regex Search failures, 
    resulting in the permanent deletion of entities from the results.

    DATA REQUIREMENTS:
    - test_data: Must be in the pre-tokenized format used during the training phase.

    EXAMPLE USAGE:
        data = []
        with open('test_dataset.jsonl', 'r') as f:
            for line in f:
                data.append(json.loads(line))
        texts = []
        for entry in data:
            patient_id = entry['id']
            text = entry['text']
            texts.append({
                    'patient_id': patient_id,
                    'text': text
                })
        label_map = {0: "O", 1: "B-ENTITY", 2: "I-ENTITY"}

        final_results = []

        for i, document in enumerate(test_data['test']):
            current_text = texts[i]['text']
            doc_results = get_ner_predictions(model, tokenizer, document, label_map, current_text)
            final_results.append({
                "text": doc_results['text'],
                "entities": doc_results['entities']
    """
    model.to(device)
    if torch.cuda.is_available():
        model.to("cuda")
    model.eval()

    all_filtered_tokens = []
    all_filtered_preds = []
    all_input_ids = []
    all_offsets = []

    for i in range(len(document['input_ids'])):
        input_ids = torch.tensor(document['input_ids'][i]).unsqueeze(0).to(device)
        attention_mask = torch.tensor(document['attention_mask'][i]).unsqueeze(0).to(device)
        token_type_ids = (
            torch.tensor(document['token_type_ids'][i]).unsqueeze(0).to(device)
            if 'token_type_ids' in document
            else None
        )

        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1).squeeze().cpu().numpy()

            input_ids = input_ids.squeeze().cpu().numpy()

            tokens = tokenizer.convert_ids_to_tokens(input_ids)
            pred_labels = [label_map[pred] for pred in predictions]

            for token, label in zip(tokens, pred_labels):
                if token not in tokenizer.all_special_tokens:
                    all_filtered_tokens.append(token)
                    all_filtered_preds.append(label)
                    input_id = tokenizer.convert_tokens_to_ids(token)
                    all_input_ids.append(input_id)
    all_input_ids = torch.tensor(all_input_ids)
    encoding = tokenizer.decode(all_input_ids.squeeze(), skip_special_tokens=False)

    text = tokenizer.convert_tokens_to_string(all_filtered_tokens)
    entities = extract_words_from_subwords(all_filtered_tokens, all_filtered_preds, current_text)
    final_entities = group_consecutive_entities(entities)

    return {
        "text": current_text,
        "tokens": all_filtered_tokens,
        "labels": all_filtered_preds,
        "entities": final_entities
    }
def extract_words_from_subwords(tokens, labels, text):
    entities = []
    current_tokens = []
    current_labels = []
    search_start = 0

    for i, (token, label) in enumerate(zip(tokens, labels)):
        if not token.startswith('##'):
            if current_tokens:
                majority_label = Counter(current_labels).most_common(1)[0][0]
                entity_text = ''.join(current_tokens)
                match = re.search(re.escape(entity_text), text[search_start:], re.ASCII)
                if match:
                    start, end = match.span()
                    start += search_start
                    end += search_start
                    entities.append({
                        "mention": entity_text,
                        "label": majority_label,
                        "start": start,
                        "end": end
                    })
                    search_start = end
                current_tokens = []
                current_labels = []

            current_tokens = [token]
            current_labels = [label]

        else:
            current_tokens.append(token[2:])
            current_labels.append(label)
    if current_tokens:
        majority_label = Counter(current_labels).most_common(1)[0][0]
        entity_text = ''.join(current_tokens)
        match = re.search(re.escape(entity_text), text[search_start:], re.ASCII)
        if match:
            start, end = match.span()
            start += search_start
            end += search_start
            entities.append({
                "mention": entity_text,
                "label": majority_label,
                "start": start,
                "end": end
            })
            search_start = end
    return entities

def group_consecutive_entities(entities):
    grouped_entities = []
    current_group = []

    for i, entity in enumerate(entities):
        if entity['label'] != 'O':
            if current_group:
                last_entity = current_group[-1]
                if (last_entity['end'] == entity['start']-1) and (
                        (last_entity['label'] == entity['label']) or
                        (last_entity['label'] == 'B-ENTITY' and entity['label'] == 'I-ENTITY')):
                    current_group.append(entity)
                else:
                    grouped_entities.append(current_group)
                    current_group = [entity]
            else:
                current_group = [entity]
        else:
            if current_group:
                grouped_entities.append(current_group)
                current_group = []
    if current_group:
        grouped_entities.append(current_group)

    final_entities = []
    for group in grouped_entities:
        merged_entity_text = ' '.join([entity['mention'] for entity in group])
        majority_label = Counter([entity['label'] for entity in group]).most_common(1)[0][0]

        start = group[0]['start']
        end = group[-1]['end']

        final_entities.append({
            "mention": merged_entity_text,
            "label": majority_label,
            "start": start,
            "end": end
        })
    return final_entities

In [ ]:
import json
def create_json_from_results(results, texts):
    json_data = {
        "system_name": "ELCardioCC_ner_baseline",
        "submission": []
    }

    for i, result in enumerate(results):
        doc_annotations = []
        for mentions in result['entities']:
            mention = mentions['mention']
            start = mentions['start']
            end = mentions['end']

            annotation = {
                "start": start,
                "end": end,
                "mention": mention,
                "code": ""
            }
            doc_annotations.append(annotation)


        json_data["submission"].append({
            "id": texts[i]['patient_id'],
            "annotations": doc_annotations
        })
    with open('data/results.json', 'w', encoding='utf-8') as f:
        json.dump(json_data, f, ensure_ascii=False, indent=4)